<a href="https://colab.research.google.com/github/AellaNandini/RL/blob/main/Ass_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Assignment 1: Policy Iteration in a Deterministic 2x2 Grid World

# 1. Define the MDP

gamma = 0.9
goal_reward = 100
normal_reward = -1
theta = 0.000001

states = ['S0', 'S1']
actions = ['Up', 'Down', 'Left', 'Right']

# Grid positions:
# S0   WALL
# S1    G

positions = {
    'S0': (0, 0),
    'S1': (1, 0),
    'WALL': (0, 1),
    'G': (1, 1)
}

# Action movements
moves = {
    'Up': (-1, 0),
    'Down': (1, 0),
    'Left': (0, -1),
    'Right': (0, 1)
}

# 2. Define transition and reward functions

def get_transition(state, action):

    row, col = positions[state]
    dr, dc = moves[action]

    new_position = (row + dr, col + dc)

    # Find destination
    destination = None

    for s, pos in positions.items():
        if pos == new_position:
            destination = s
            break

    # Wall or boundary: stay in current state
    if destination is None or destination == 'WALL':
        return state, normal_reward

    # Goal reached
    if destination == 'G':
        return 'G', goal_reward

    # Normal movement
    return destination, normal_reward


# Display transition information
print("STATE, ACTION, NEXT STATE AND REWARD")
print("------------------------------------")

for state in states:
    for action in actions:
        next_state, reward = get_transition(state, action)
        print(state, action, "->", next_state,
              "Reward:", reward)


# 3. Initialize policy and state values

policy = {
    'S0': 'Up',
    'S1': 'Up'
}

V = {
    'S0': 0.0,
    'S1': 0.0,
    'G': 0.0
}

print("\nInitial Policy:", policy)
print("Initial Values:", V)


# 4. Policy Evaluation

def policy_evaluation(policy, V):

    iteration = 0

    while True:

        delta = 0
        new_V = V.copy()
        iteration += 1

        for state in states:

            action = policy[state]

            next_state, reward = get_transition(
                state, action
            )

            new_V[state] = (
                reward + gamma * V[next_state]
            )

            delta = max(
                delta,
                abs(new_V[state] - V[state])
            )

        V = new_V

        if delta < theta:
            break

    return V, iteration


# 5. Calculate action values

def calculate_Q(state, action, V):

    next_state, reward = get_transition(
        state, action
    )

    return reward + gamma * V[next_state]


# 6. Policy Iteration

policy_iteration = 0

while True:

    policy_iteration += 1

    print("\n========== POLICY ITERATION",
          policy_iteration, "==========")

    # Policy Evaluation
    V, eval_iterations = policy_evaluation(policy, V)

    print("Policy:", policy)
    print("Evaluation iterations:", eval_iterations)

    print("State Values:")
    for state in states:
        print(state, "=", round(V[state], 6))

    # Policy Improvement
    stable = True
    new_policy = policy.copy()

    for state in states:

        old_action = policy[state]

        q_values = {}

        for action in actions:
            q_values[action] = calculate_Q(
                state, action, V
            )

        best_action = max(q_values, key=q_values.get)

        new_policy[state] = best_action

        if best_action != old_action:
            stable = False

    print("Improved Policy:", new_policy)

    policy = new_policy

    # Check policy stability
    if stable:
        print("\nOptimal Policy Found!")
        break


# 7. Display Final Optimal State Values

print("\n========== FINAL OPTIMAL STATE VALUES ==========")

for state in states:
    print(state, "=", round(V[state], 4))

print("G = 0 (Terminal State)")


# 8. Display Final Action-Value Table

print("\n========== FINAL ACTION VALUE TABLE ==========")

print(f"{'State':<10}", end="")

for action in actions:
    print(f"{action:<12}", end="")

print()

for state in states:

    print(f"{state:<10}", end="")

    for action in actions:
        q = calculate_Q(state, action, V)
        print(f"{q:<12.4f}", end="")

    print()


# 9. Display Optimal Policy

print("\n========== OPTIMAL POLICY ==========")

for state in states:
    print(state, ":", policy[state])

print("G : Terminal")


# 10. Bellman Optimality Verification

print("\n========== BELLMAN OPTIMALITY VERIFICATION ==========")

for state in states:

    q_values = [
        calculate_Q(state, action, V)
        for action in actions
    ]

    max_q = max(q_values)

    print(
        state,
        "V*(s) =", round(V[state], 4),
        "Max Q*(s,a) =", round(max_q, 4)
    )

    assert abs(V[state] - max_q) < 0.0001

print("\nBellman Optimality Verified Successfully!")


STATE, ACTION, NEXT STATE AND REWARD
------------------------------------
S0 Up -> S0 Reward: -1
S0 Down -> S1 Reward: -1
S0 Left -> S0 Reward: -1
S0 Right -> S0 Reward: -1
S1 Up -> S0 Reward: -1
S1 Down -> S1 Reward: -1
S1 Left -> S1 Reward: -1
S1 Right -> G Reward: 100

Initial Policy: {'S0': 'Up', 'S1': 'Up'}
Initial Values: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

========== POLICY ITERATION 1 ==========
Policy: {'S0': 'Up', 'S1': 'Up'}
Evaluation iterations: 133
State Values:
S0 = -9.999992
S1 = -9.999992
Improved Policy: {'S0': 'Up', 'S1': 'Right'}

========== POLICY ITERATION 2 ==========
Policy: {'S0': 'Up', 'S1': 'Right'}
Evaluation iterations: 2
State Values:
S0 = -9.999993
S1 = 100.0
Improved Policy: {'S0': 'Down', 'S1': 'Right'}

========== POLICY ITERATION 3 ==========
Policy: {'S0': 'Down', 'S1': 'Right'}
Evaluation iterations: 2
State Values:
S0 = 89.0
S1 = 100.0
Improved Policy: {'S0': 'Down', 'S1': 'Right'}

Optimal Policy Found!

========== FINAL OPTIMAL STATE VALUES =======